[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_04_Regression/03_regularization.ipynb)

# Episode 10 – Regularization: Ridge, Lasso & ElasticNet

**Machine Learning Bootcamp** | Module 04

---

## 🎯 Learning Objectives
- Understand L1 and L2 regularisation and why they help
- Apply Ridge, Lasso, and ElasticNet regression
- Tune the regularisation strength (alpha) with cross-validation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

sns.set_theme(style='whitegrid')

## 1. Why Regularisation?

Regularisation adds a **penalty term** to the loss function to shrink coefficients and prevent overfitting:

| Method | Loss | Effect |
|--------|------|--------|
| **Ridge** (L2) | MSE + α Σθᵢ² | Shrinks all coefficients |
| **Lasso** (L1) | MSE + α Σ\|θᵢ\| | Shrinks + sets some to **zero** (feature selection) |
| **ElasticNet** | MSE + α(L1 + L2 mix) | Combination of both |

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

models = {
    'Ridge (α=1.0)':      Ridge(alpha=1.0),
    'Ridge (α=10.0)':     Ridge(alpha=10.0),
    'Lasso (α=0.1)':      Lasso(alpha=0.1),
    'ElasticNet (α=0.1)': ElasticNet(alpha=0.1, l1_ratio=0.5),
}

print(f'{"Model":<25}  RMSE (test)')
print('-' * 38)
for name, m in models.items():
    m.fit(X_train_s, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test_s)))
    print(f'{name:<25}  {rmse:.4f}')

## 2. Lasso for Feature Selection

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=42)
lasso_cv.fit(X_train_s, y_train)
print(f'Best alpha: {lasso_cv.alpha_:.4f}')

coef_df = {
    'feature': housing.feature_names,
    'coefficient': lasso_cv.coef_
}
import pandas as pd
coef_df = pd.DataFrame(coef_df).sort_values('coefficient', key=abs, ascending=False)

plt.figure(figsize=(9, 5))
colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(0, color='black', lw=0.8)
plt.title('Lasso Coefficients (α = LassoCV best)')
plt.xlabel('Coefficient value')
plt.tight_layout(); plt.show()
print('Zero coefficients (dropped features):', (lasso_cv.coef_ == 0).sum())

## 🏋️ Exercises

1. Use `RidgeCV` to automatically find the best alpha. Compare to the manual values above.
2. Create an overfitted polynomial model (degree 8) and show that Ridge regularisation reduces test RMSE.
3. Vary `l1_ratio` in `ElasticNet` from 0 (Ridge) to 1 (Lasso). Plot the number of non-zero coefficients vs. `l1_ratio`.

---
**Next ▶ [Module 05 – Classification](../Module_05_Classification/01_logistic_regression.ipynb)**